TODO :
- Decide what one row represents and write it down (for example: a repository, a film, a weather station, or an article). As Exercises 6 and 7 showed, this choice determines what your clusters mean.

    - A row is a game or stream. Our features are viewer counts, tag one-hot, hour-of-day profile

- Build the feature table X. The table below suggests features for each project source. Use one-hot encoding for categories, take the log of skewed counts, and use TF-IDF for text columns.

- Standardise the features ((feat - feat.mean()) / feat.std()), then cluster for two or three values of 𝑘. If you compute silhouette scores on more than a few thousand rows, pass sample_size=10_000 to silhouette_score so that the 𝑂(𝑛2) pairwise distance calculation does not hang.

- Print the cluster profile table and give every cluster a short descriptive name. Put the cluster names and interpretation in your report, and the full profile table in the appendix.

- Inspect any small or hard-to-name cluster. As with the 14 broken-meter trips in Exercise 6, an unusual cluster often reveals a data-quality issue missed in the previous weeks.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.width", 100)
sns.set_theme(style="whitegrid")

CLEAN_DATA_DIR = Path.cwd().parent / "data/clean/"

clean_games_path = CLEAN_DATA_DIR / "games_cleaned.parquet"
clean_clips_path = CLEAN_DATA_DIR / "clips_cleaned.parquet"
clean_channels_path = CLEAN_DATA_DIR / "channels_cleaned.parquet"

cgdf = pd.read_parquet(clean_games_path)
cdf = pd.read_parquet(clean_clips_path)
chdf = pd.read_parquet(clean_channels_path)

### Feature selection

First we decide what one row represents. In our case we have 3 different data frames, we can join them together to get a very rich representation of each clip.

Features we will include : game_name, clip_language, clip_title, view_count, duration, is_featured, time posted, is branded, content classification labels, tags, content_classification_labels

## Building the feature table

### Content Classification
Here, we use one-hot encoding for the different labels present in content classification column.

In [2]:
# One-hot encoding and bag of words
# one-hot encode the content_classification_labels column
import re

# extract non-empty labels
label_set = {
    label
    for text in chdf['content_classification_labels'].dropna()
    for label in re.findall(r"'([^']+)'", str(text))
    if label.strip() not in ("[]", "")
}

for label in label_set:
    chdf[f"label_{label}"] = (
        chdf["content_classification_labels"]
        .astype("string")
        .str.contains(f"'{label}'", regex=False, na=False)
        .astype("int8")
    )

# label_none = 1 if all category columns sum to 0
label_cols = [f"label_{label}" for label in label_set]
chdf["label_none"] = (chdf[label_cols].sum(axis=1) == 0).astype("int8")

chdf = chdf.drop(columns=["content_classification_labels"])

print(chdf[:5])

   broadcaster_id broadcaster_language  game_id          game_name  \
0       234973404                   en   509658      Just Chatting   
1       683970274                   en   509658      Just Chatting   
2       552120296                   en    18122  World of Warcraft   
3       423486275                   ru   509658      Just Chatting   
4           12826                   en   509658      Just Chatting   

                                               title  \
0  IRL Shanghai 🇨🇳 I'm back with @pokimane 🥰 Dres...   
1  SOUL FOOD COOKING - FRIED CHICKEN, MAC N CHEES...   
2  [DROPS ON] UNBANNED WOW FOREVER BETA FINAL DAY...   
3  ПОДГОТОВКА К ГРЯЗНОЙ ФОТОСЕССИИ С ДЕВОЧКАМИ)) ...   
4  Twitch Public Access (October 2026) | Celebrat...   

                                                tags  is_branded_content  label_SexualThemes  \
0  [['Travel' 'food' 'lifestyle' 'Chinese中文' 'IRL...               False                   0   
1  [['american' 'girl' 'asian' 'canadian' 'Englis.

### Tags

Keep in mind that the most represented tags are language tags. We could try to remove the language tags to fix this issue. 

The probelem is that if we add 400 tags, the clustering will be overwhelmingly driven by the tags and not the other columns. We need to make a choice about which tags we will represent. It could make sense to include only the 30 most popular tags. 

Another issue remains : tags are often language based: a german game has german language tags , same for russian, english etc. The tag for 'win' will be 'ganar' in spanish channels, 'vincere' in italian, and so on, with many different syntaxes for the same concept. This is bad because we already have a language variable, we want the tag concept as a source of new information.

This opens up a lot of other problems, namely that non-English tags are mutually exclusive across language communities, the clusters will naturally group rows by the language or even alphabet of their tags.

In [6]:
import re

# 1. Properly extract individual tags from the stringified list format
def parse_tags(val):
    if val is None or pd.isna(val):
        return []
    # Extract everything inside single quotes: 'tag'
    return [t.strip().lower() for t in re.findall(r"'([^']+)'", str(val)) if t.strip()]

chdf["clean_tags"] = chdf["tags"].apply(parse_tags)

# 2. Define language tags to exclude (lowercase)
lang_tags_to_remove = {
    "english", "русский", "español", "日本語", "deutsch", "français",
    "português", "italiano", "polski", "türkçe", "한국어", "chinese中文",
    "中文", "ukr", "українська", "عربي", "čeština", "magyar", "suomi"
}

# Remove language tags
chdf["concept_tags"] = chdf["clean_tags"].apply(
    lambda tags: [t for t in tags if t not in lang_tags_to_remove]
)

# 3. Check the TRUE unique tag count and top 30 concept tags
exploded_concepts = chdf["concept_tags"].explode()
print(f"Total unique non-language tags: {exploded_concepts.nunique()}")
print("\nTop 50 Concept Tags:")
print(exploded_concepts.value_counts().head(50))

Total unique non-language tags: 1539

Top 50 Concept Tags:
concept_tags
dropsenabled       135
drops有効             34
vtuber              33
dropsвключены       31
irl                 28
adhd                24
fps                 23
funny               21
chill               19
gaming              19
启用掉宝                19
ストグラ                18
العربية             17
girl                16
pvp                 16
esports             16
anime               16
warzone             15
ranked              15
dropsactivados      15
deadlock            14
drops               13
feet                13
arcraiders          13
slots               12
overwatch           12
osrs                12
fortnite            12
nopixel             11
nopixelv            11
gta                 11
competitive         10
egirl               10
cs2                 10
雑談                  10
arabic              10
callofduty           8
educational          8
cod                  8
minecraft            8
calldofd

### Join into one single feature df X

## Standardize X


## Clustering